# 01c · 흡수 확률과 도박꾼의 파산: 기본행렬 (Absorption and Gambler's Ruin: The Fundamental Matrix)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.8 (탈출 분포, 도박꾼 파산의 $h(i)$ 차분방정식) · §1.9 (기대 탈출시간) · Ross 12e §4.5.1 (도박꾼 파산 $P_i$ 유도) · §4.6 (일시 상태 평균 체류시간 $S=(I-P_T)^{-1}$) · Norris §1.3 (도달확률·평균 도달시간 = 최소 비음해) · Lawler 2e §1.5 ($M=(I-Q)^{-1}$의 원소 = 기대 방문 횟수) |
| 선수 노트북 | 01b, 01a, 00a |
| 예상 소요 | 90분 |
| 상태 | draft |

01b에서 일시 상태에서 출발하면 확률 1로 어떤 닫힌 클래스에 흡수된다는 것을 보였다. 이 노트북은 그 다음 두 질문 — **어느** 닫힌 클래스에, **얼마 만에** 흡수되는가 — 를 행렬 하나로 푼다. 정준형 $P=\begin{pmatrix}Q&R\\0&I\end{pmatrix}$의 일시 블록 $Q$에서 **기본행렬(fundamental matrix)** $N=(I-Q)^{-1}$을 만들면 흡수확률은 $B=NR$, 기대 흡수시간은 $t=N\mathbf 1$이다. 00a에서 시뮬레이션으로만 봤던 도박꾼 파산의 승률 $0.268$과 게임 길이 $23.2$걸음이 이제 닫힌 공식과 행렬 곱, 두 가지 방법으로 나온다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_power
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (is_stochastic, simulate_chain, classify_states, absorption, hitting_times,
                          gamblers_ruin_matrix, gamblers_ruin_prob, gamblers_ruin_duration)
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("01c")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01b) 재귀 상태와 일시 상태의 정의를 귀환확률 $\rho_{xx}$로 쓰시오.

<details><summary>정답</summary>

$\rho_{xx}=P_x(T_x<\infty)$, $T_x=\min\{n\ge1:X_n=x\}$. 재귀 $\iff\rho_{xx}=1$, 일시 $\iff\rho_{xx}<1$.

</details>

**Q2.** (01b) 유한 사슬에서 닫힌 소통 클래스와 닫히지 않은 클래스는 각각 재귀/일시 중 무엇인가? 이유 한 줄.

<details><summary>정답</summary>

닫힌 클래스 = 재귀(갇혀 있으므로 유한 개의 상태 중 어딘가에 무한히 자주 돌아온다), 새는 클래스 = 일시(한 번 나가면 돌아올 수 없으므로 $\rho_{xx}<1$).

</details>

**Q3.** (01b) 일시 상태 $x$에서 출발할 때 방문 횟수 $V_x$(출발 포함)의 분포와 평균은?

<details><summary>정답</summary>

기하분포 $P_x(V_x=k)=(1-\rho_{xx})\rho_{xx}^{k-1}$, $E_xV_x=\dfrac1{1-\rho_{xx}}=\sum_{n\ge0}p^n(x,x)$. 오늘 이 값이 기본행렬의 대각 원소 $N_{xx}$임을 본다.

</details>

**Q4.** (01a) 2-상태 사슬 $\begin{pmatrix}1-a&a\\b&1-b\end{pmatrix}$의 $p^n(0,0)$ 공식은?

<details><summary>정답</summary>

$p^n(0,0)=\dfrac{b}{a+b}+\dfrac{a}{a+b}(1-a-b)^n$; 두 번째 고유값 $\lambda_2=1-a-b$가 수렴 속도를 정한다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **흡수 사슬의 정준형(canonical form)**: 일시 상태 집합 $T$와 흡수(또는 닫힌) 상태 집합 $A$로 상태를 재배열하면
  $$P=\begin{pmatrix}Q&R\\0&I\end{pmatrix},\qquad Q=P[T,T]\ (|T|\times|T|),\quad R=P[T,A]\ (|T|\times|A|).$$
  닫힌 클래스가 한 상태가 아니면(01b의 $\{4,5,6\}$) 그 클래스를 통째로 흡수 상태 하나로 뭉쳐도(lumping) 흡수확률·흡수시간은 변하지 않는다 — 일시 상태에서는 '클래스에 들어갔는가'만 중요하기 때문이다.
- **기본행렬(fundamental matrix)** $N=(I-Q)^{-1}$. 의미: $N_{ij}=E_i[V_j]$, $V_j=\sum_{n\ge0}\mathbf 1\{X_n=j\}$ = 흡수 전에 일시 상태 $j$를 방문하는 횟수(출발 포함). 코드: `absorption(P, transient, absorbing)["N"]`.
- **흡수확률 행렬** $B=NR$ ($|T|\times|A|$): $B_{ia}=P_i(X_T=a)$ = $A$에 **처음 들어갈 때** 있는 상태가 $a$일 확률(흡수 상태가 한 점이면 그냥 '$a$에 흡수될 확률'). **기대 흡수시간** $t=N\mathbf 1$: $t_i=E_i[T]$, $T=\min\{n\ge0:X_n\in A\}$.
- **1단계 분석(first-step analysis)**: 미지량 $h(x)$(확률 또는 기대시간)를 첫 걸음 $X_1$에 조건화해 선형방정식으로 만드는 기법. 확률이면 $h(x)=\sum_yp(x,y)h(y)$, 시간이면 $h(x)=1+\sum_yp(x,y)h(y)$ — 시간에는 '방금 뗀 한 걸음'의 상수항 1이 붙는다. 행렬로 쓰면 각각 $h=Qh+R\mathbf 1_a$, $t=\mathbf 1+Qt$.
- **도박꾼 파산(gambler's ruin)**: 상태 $\{0,\dots,N\}$, 안쪽 $0<i<N$에서 $p(i,i+1)=p$, $p(i,i-1)=q=1-p$, $0$과 $N$은 흡수. $h(i)=P_i(\text{reach }N\text{ before }0)$, $\tau(i)=E_i[T]$. 코드: `gamblers_ruin_matrix`, `gamblers_ruin_prob`, `gamblers_ruin_duration`.

### 2.2 정리 1 — 기본행렬의 존재와 의미 (Lawler 2e §1.5, Ross 12e §4.6)

> 유한 사슬에서 $T$가 일시 상태 전체이면 $Q^n\to0$, $I-Q$는 가역이고
> $$N=(I-Q)^{-1}=\sum_{k\ge0}Q^k,\qquad N_{ij}=E_i[V_j]\quad(i,j\in T).$$

**가정이 왜 필요한가**
- $T$가 **정말 일시 상태만** 포함: $T$ 안에 닫힌 클래스가 숨어 있으면 그 블록의 $Q$는 행합이 1인 부분행렬을 품어 고유값 1을 가지므로 $I-Q$가 특이(singular)행렬이 된다 — 역행렬이 없다(E3에서 확인).
- **유한성**: 각 일시 상태에서 $|T|$걸음 안에 $T$를 떠날 확률이 어떤 양수 $\epsilon$ 이상이어야 $\|Q^{k|T|}\|\le(1-\epsilon)^k\to0$이 성립한다. 무한 상태공간에서는 $N=\sum Q^k$가 여전히 기대 방문 횟수 행렬이지만(단조수렴), $P_i(T<\infty)<1$일 수 있어 $Q^n$의 행합이 0으로 가지 않고, 선형방정식 $B=QB+R$, $t=\mathbf 1+Qt$의 해가 유일하지 않아 '최소 비음해'를 택해야 한다(Norris §1.3, 01x).

<details><summary>증명</summary>

**(1) $Q^n\to0$.** 유한 사슬에서 일시 상태 $i$는 어떤 닫힌 클래스에 도달 가능하고(01b 정리 3), 최단 경로는 상태를 반복하지 않으므로 길이가 $|T|$ 이하다. 따라서 $P_i(T\le|T|)>0$이고, 일시 상태가 유한 개이므로
$$\epsilon:=\min_{i\in T}P_i(T\le|T|)>0.$$
마르코프 성질로 $|T|$걸음씩 끊어 보면, 매 구간마다 어디에 있든 흡수될 확률이 $\epsilon$ 이상이므로 $P_i(T>k|T|)\le(1-\epsilon)^k$. 한편 $(Q^n)_{ij}=P_i(X_n=j,\ n<T)$ ($j\in T$이면 시각 $n$까지 흡수되지 않았다는 뜻이 자동으로 포함되므로 $=P_i(X_n=j)$)이고, 행합 $\sum_{j\in T}(Q^n)_{ij}=P_i(T>n)$. 그러므로 $Q^{k|T|}$의 모든 행합이 $(1-\epsilon)^k\to0$, 즉 $Q^n\to0$ (행합은 $n$에 대해 단조감소).

**(2) 급수와 역행렬.** 망원합 $(I-Q)\sum_{k=0}^{n}Q^k=I-Q^{n+1}\to I$. 각 $Q^k$의 원소는 비음이고 행합 $P_i(T>k)$가 기하급수적으로 줄므로 $\sum_kQ^k$는 성분별로 수렴한다. 극한을 취하면 $(I-Q)\sum_{k\ge0}Q^k=I$이므로 $I-Q$는 가역이고 $N=(I-Q)^{-1}=\sum_{k\ge0}Q^k$.

**(3) 방문 횟수.** $V_j=\sum_{n\ge0}\mathbf 1\{X_n=j\}$의 각 항이 비음이므로 단조수렴으로 기댓값과 합을 바꿀 수 있다:
$$E_i[V_j]=\sum_{n\ge0}P_i(X_n=j)=\sum_{n\ge0}(Q^n)_{ij}=N_{ij}.\qquad\square$$

</details>

### 2.3 정리 2 — 흡수확률과 기대 흡수시간 (Durrett 3e §1.8–§1.9, Norris §1.3)

> $B=NR$, $t=N\mathbf 1$. 동치로, $B$는 $B=QB+R$의 유일해이고 $t$는 $t=\mathbf 1+Qt$의 유일해다.

**가정이 왜 필요한가**
- $I-Q$ 가역(정리 1): 1단계 분석은 방정식만 줄 뿐 해가 하나라는 보장은 주지 않는다. 유한 사슬에서는 가역성이 유일성을 준다. 무한 상태공간에서는 $h=Qh+R\mathbf 1_a$의 해가 여러 개일 수 있어 '최소 비음해'를 택해야 한다 (Norris §1.3 Thm 1.3.2, 1.3.5).

<details><summary>증명</summary>

**흡수확률.** $i\in T$, $a\in A$에 대해 첫 걸음에 조건화한다. 첫 걸음에 바로 $a$로 가거나($p(i,a)$), 다른 흡수 상태로 가서 영영 $a$에 못 가거나(기여 0), 일시 상태 $j$로 가서 거기서 새로 시작한다:
$$B_{ia}=p(i,a)+\sum_{j\in T}p(i,j)B_{ja}\quad\Longleftrightarrow\quad B=R+QB\quad\Longleftrightarrow\quad(I-Q)B=R.$$
$I-Q$가 가역이므로 $B=(I-Q)^{-1}R=NR$이 유일해다.

**기대 흡수시간 (방법 1: 1단계 분석).** $i\in T$에서 한 걸음을 떼면 시간이 1 흐르고, 흡수 상태에 닿았으면 끝(남은 시간 0), 일시 상태 $j$에 닿았으면 남은 시간의 기댓값은 $t_j$:
$$t_i=1+\sum_{j\in T}p(i,j)t_j\quad\Longleftrightarrow\quad(I-Q)t=\mathbf 1\quad\Longleftrightarrow\quad t=N\mathbf 1.$$

**(방법 2: 방문 횟수의 합).** 흡수 전 시각 $0,1,\dots,T-1$에는 정확히 하나의 일시 상태에 있으므로 $T=\sum_{j\in T}V_j$. 정리 1로 $E_iT=\sum_{j\in T}N_{ij}=(N\mathbf 1)_i$. 두 방법이 같은 답을 주는 것은 $(I-Q)N=I$의 양변에 $\mathbf 1$을 곱한 것과 같다. $\square$

</details>

### 2.4 정리 3 — 도박꾼 파산의 닫힌 공식 (Durrett 3e §1.8 Example 1.43 부근, Ross 12e §4.5.1)

> $r=q/p$라 하자. $p\ne\tfrac12$이면
> $$h(i)=\frac{1-r^i}{1-r^N},\qquad \tau(i)=\frac{i}{q-p}-\frac{N}{q-p}\cdot\frac{1-r^i}{1-r^N},$$
> $p=\tfrac12$이면 $h(i)=i/N$, $\tau(i)=i(N-i)$.

**가정이 왜 필요한가**
- $0<p<1$: 차분방정식 $p\,h(i+1)-h(i)+q\,h(i-1)=0$의 특성근이 $1$과 $r=q/p$로 잘 정의된다. $p=\tfrac12$이면 $r=1$이 **중근**이라 일반해가 $A+Br^i$ 대신 $A+Bi$로 바뀌므로 따로 다룬다.
- 경계조건 $h(0)=0,\ h(N)=1$, $\tau(0)=\tau(N)=0$: 흡수 상태가 정확히 두 개라는 뜻. 일반해의 상수 두 개를 결정하려면 경계조건도 정확히 두 개여야 한다.

<details><summary>증명</summary>

**승률 $h$.** 1단계 분석: $0<i<N$에서 $h(i)=p\,h(i+1)+q\,h(i-1)$. $h(i)=(p+q)h(i)$로 쓰고 정리하면
$$p\,[h(i+1)-h(i)]=q\,[h(i)-h(i-1)]\quad\Longrightarrow\quad h(i+1)-h(i)=r\,[h(i)-h(i-1)].$$
즉 연속한 차분이 공비 $r$의 등비수열: $h(i)-h(i-1)=r^{i-1}[h(1)-h(0)]=r^{i-1}h(1)$. $i=1..k$에 대해 더하면 $h(k)=h(1)\sum_{i=0}^{k-1}r^i=h(1)\dfrac{1-r^k}{1-r}$ ($r\ne1$). $h(N)=1$에서 $h(1)=\dfrac{1-r}{1-r^N}$이므로 $h(i)=\dfrac{1-r^i}{1-r^N}$. $p=\tfrac12$($r=1$)이면 차분이 상수 $h(1)$이라 $h(k)=k\,h(1)$, $h(N)=1$에서 $h(i)=i/N$.

**기대 게임 길이 $\tau$.** $0<i<N$에서 $\tau(i)=1+p\,\tau(i+1)+q\,\tau(i-1)$ — 비제차 선형 차분방정식. 동차해는 위와 같이 $A+Br^i$ ($p\ne\tfrac12$) 또는 $A+Bi$ ($p=\tfrac12$).
특수해: $p\ne\tfrac12$이면 $\tau_p(i)=ci$를 대입하면 $ci=1+pc(i+1)+qc(i-1)=1+ci+c(p-q)$이므로 $c=1/(q-p)$. $p=\tfrac12$이면 $\tau_p(i)=-i^2$: $-i^2=1-\tfrac12(i+1)^2-\tfrac12(i-1)^2=1-i^2-1$. ✓
일반해 $\tau(i)=\dfrac{i}{q-p}+A+Br^i$에 $\tau(0)=0$, $\tau(N)=0$을 넣으면 $A=-B$, $\dfrac{N}{q-p}+B(r^N-1)=0$, 즉 $B=\dfrac{N}{(q-p)(1-r^N)}$, 따라서
$$\tau(i)=\frac{i}{q-p}-\frac{N}{q-p}\cdot\frac{1-r^i}{1-r^N}=\frac{i-N\,h(i)}{q-p}.$$
$p=\tfrac12$: $\tau(i)=-i^2+A+Bi$, $\tau(0)=0\Rightarrow A=0$, $\tau(N)=0\Rightarrow B=N$, 즉 $\tau(i)=i(N-i)$. $\square$

</details>

### 2.5 직관

$N=\sum_kQ^k$는 '흡수되기 전에 일시 상태들을 몇 번이나 지나가는가'를 세는 행렬이다. 흡수확률 $NR$은 (일시 상태 $j$를 헤매는 횟수) × ($j$에서 한 걸음에 $a$로 흡수될 확률)을 모든 $j$에 대해 더한 것이고, 흡수시간 $N\mathbf 1$은 헤매는 횟수의 총합이다. 1단계 분석은 같은 내용을 '첫 걸음을 떼고 나면 같은 문제가 다시 시작된다'는 자기유사성으로 쓴 것이다 — 확률 방정식에는 상수항이 없고 시간 방정식에는 방금 뗀 한 걸음 $1$이 붙는다.

도박꾼 파산에서 $p=0.45$라는 5% 불리함이 $i=5,\ N=10$에서 승률을 $0.5\to0.268$로 떨어뜨리는 이유는 차분 $h(i)-h(i-1)$이 공비 $r=11/9$로 **커지는** 등비수열이기 때문이다: 승률은 목표 근처에서만 가파르게 오르고 출발점 근처에서는 납작하다. 게임 길이는 $p=\tfrac12$의 포물선 $i(N-i)$보다 조금 짧아진다(끝나는 쪽으로 밀리므로).

### 2.6 함정(traps)

1. `absorption(P, transient, absorbing)`의 `transient`에 재귀 상태를 넣으면 $I-Q$가 특이행렬이 되어 `inv`가 `LinAlgError`를 내거나(정확히 특이) 쓰레기 값을 준다(수치적으로 거의 특이). 먼저 `classify_states`로 일시 상태를 확인할 것.
2. 시뮬레이션으로 $E[T]$를 추정할 때 지평 `n_steps`를 짧게 잡으면 흡수 안 된 경로의 $T$가 `inf`로 남아 평균이 정의되지 않고, 이를 지평 값으로 바꿔 평균 내면 **과소추정**된다(E3). 흡수 안 된 비율을 항상 출력하고, $Q^n$의 행합으로 $P(T>n)$이 무시할 만한지 확인할 것.
3. $h(i)$ 공식의 $r=q/p$는 '질 확률 / 이길 확률'. $p>\tfrac12$이면 $r<1$이고 분모·분자가 모두 양수라 공식은 그대로 성립한다.
4. $p=\tfrac12$일 때 $p\ne\tfrac12$ 공식에 $p=0.5$를 대입하면 $0/0$. 코드에 `p == 0.5` 분기가 있는 이유이고, 수학적으로는 $r\to1$ 극한(로피탈)이 $i/N$을 준다.
5. 닫힌 클래스가 여러 상태(01b의 $\{4,5,6\}$)이면 흡수 집합 $A$에 그 상태들을 **모두** 넣거나 하나로 뭉쳐야 한다. 일부만 넣으면 $R$의 열이 빠져 $B$의 행합이 1이 안 된다.

✍️ **내 말로**: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) $N_{ij}$의 확률적 의미, (2) $B=NR$이 왜 성립하는지, (3) 1단계 분석에서 '확률' 방정식과 '시간' 방정식의 차이(상수항 1의 유무)를 각각 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

`None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 앞의 두 값은 00a에서 시뮬레이션했던 값이니 기억을 더듬어 보세요. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # N=10, p=0.45, i=5 에서 목표 N 에 먼저 도달할 확률 h(5) 는? (00a 에서 시뮬레이션했던 값)
    "p_win_045": None,
    # 같은 설정에서 게임이 끝날 때까지의 기대 걸음 수 tau(5) 는? (p=1/2 이면 5*5 = 25)
    "duration_045": None,
    # p=1/2 일 때 5 에서 출발해 흡수 전에 상태 5 를 방문하는 횟수(출발 포함)의 기댓값 N_55 는?
    "n55_half": None,
    # 공정한 동전을 던져 처음으로 'HH' 가 연속으로 나올 때까지의 기대 던짐 횟수는?
    # (상태: 0 = 진행 없음, 1 = 마지막이 H, 2 = HH 달성)
    "t_HH": None,
    # 01b 의 7-상태 사슬에서 2 에서 출발해 클래스 {0,1} 에 흡수될 확률은?
    "p_class01_from_2": None,
    # 같은 사슬에서 2 에서 출발해 일시 집합 {2,3} 을 떠날 때까지의 기대 걸음 수는?
    "t_exit_from_2": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 정확식 vs 정확식: 기본행렬이 닫힌 공식을 재현하는가

$N=10$, $p\in\{0.45,\ 0.5\}$. `absorption`이 주는 $B=NR$의 둘째 열(흡수 상태 10)과 $t=N\mathbf 1$을 정리 3의 닫힌 공식과 `np.allclose`로 비교한다. 여기에는 난수가 없다 — 두 정확한 계산이 $10^{-12}$ 안에서 같아야 한다. 먼저 `classify_states`로 일시 집합이 정말 $\{1,\dots,9\}$인지 확인한다(함정 1).

In [ ]:
N_GR = 10
TRANS = list(range(1, N_GR))          # 일시 상태 1..9
ABS = [0, N_GR]                       # 흡수 상태
P045 = gamblers_ruin_matrix(N_GR, 0.45)
P05 = gamblers_ruin_matrix(N_GR, 0.5)
assert is_stochastic(P045) and classify_states(P045)["transient_states"] == TRANS

res045 = absorption(P045, TRANS, ABS)
res05 = absorption(P05, TRANS, ABS)
h_closed = {p: np.array([gamblers_ruin_prob(i, N_GR, p) for i in TRANS]) for p in (0.45, 0.5)}
tau_closed = {p: np.array([gamblers_ruin_duration(i, N_GR, p) for i in TRANS]) for p in (0.45, 0.5)}

assert np.allclose(res045["B"][:, 1], h_closed[0.45]) and np.allclose(res045["t"], tau_closed[0.45])
assert np.allclose(res05["B"][:, 1], h_closed[0.5]) and np.allclose(res05["t"], tau_closed[0.5])
assert np.allclose(res05["t"], [i * (N_GR - i) for i in TRANS]) and np.isclose(res05["N"][4, 4], 5.0)
print("h(i), p=0.45 :", np.round(res045["B"][:, 1], 4))
print("tau(i), p=0.45:", np.round(res045["t"], 3))
print("tau(i), p=0.5 :", np.round(res05["t"], 3), "   N_55 =", round(res05["N"][4, 4], 6))
print("row sums of B (must be 1):", np.round(res045["B"].sum(1), 12).min(), "...")

대칭 걷기($p=\tfrac12$)의 기본행렬은 닫힌 꼴 $N_{ij}=2\min(i,j)\,(N-\max(i,j))/N$을 갖는다(경계 조건이 두 개인 이산 그린 함수(Green's function)). $i=j=5$이면 $2\cdot5\cdot5/10=5$.

In [ ]:
ii, jj = np.meshgrid(TRANS, TRANS, indexing="ij")
N_half_formula = 2 * np.minimum(ii, jj) * (N_GR - np.maximum(ii, jj)) / N_GR
assert np.allclose(res05["N"], N_half_formula)
print("N (p=1/2), rows/cols = states 1..9:")
print(np.round(res05["N"], 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for p, res, c in [(0.45, res045, "C0"), (0.5, res05, "C1")]:
    axes[0].plot(TRANS, h_closed[p], color=c, lw=1.5, label=f"closed form, p={p}")
    axes[0].plot(TRANS, res["B"][:, 1], "o", color=c, mfc="white", label=f"absorption() B, p={p}")
    axes[1].plot(TRANS, tau_closed[p], color=c, lw=1.5, label=f"closed form, p={p}")
    axes[1].plot(TRANS, res["t"], "o", color=c, mfc="white", label=f"absorption() t, p={p}")
axes[0].set_xlabel("starting fortune i"); axes[0].set_ylabel("$P_i$(reach N before 0)")
axes[1].set_xlabel("starting fortune i"); axes[1].set_ylabel("$E_i[T]$ (steps)")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.suptitle("Gambler's ruin (N=10): fundamental matrix vs closed form", y=1.02)
plt.tight_layout(); plt.show()

### 4.2 MC 검증 — 00a의 두 번째 풀이

$i=5$에서 `N_PATHS`개 경로를 300걸음 시뮬레이션한다. 먼저 몇 개의 경로를 그려 '흡수'가 무엇인지 눈으로 본다: 각 경로는 $0$ 또는 $10$에 닿는 순간 멈추고, 그 시각이 $T$다.

In [ ]:
N_STEPS = 300
paths = simulate_chain(P045, 5, N_STEPS, rng, n_paths=N_PATHS)        # p = 0.45
T045 = hitting_times(paths, {0, N_GR})
assert np.isfinite(T045).all()          # 흡수 안 된 경로 없음 (다음 셀에서 정확값과 대조)

fig, ax = plt.subplots()
for k in range(12):
    stop = int(T045[k])
    ax.plot(range(stop + 1), paths[k, :stop + 1], lw=1, alpha=0.8)
    ax.plot(stop, paths[k, stop], "k|", ms=8)
ax.axhline(0, color="k", lw=0.8); ax.axhline(N_GR, color="k", lw=0.8)
ax.set_xlabel("step n"); ax.set_ylabel("fortune $X_n$")
ax.set_title("Gambler's ruin paths from i=5 (p=0.45); tick = absorption time T")
plt.show()

지평 300이 충분한지 두 가지로 확인한다: 시뮬레이션에서 흡수 안 된 경로의 비율(0이어야 정상)과, 행렬로 계산한 정확값 $P_5(T>300)=\sum_{y\in T}(Q^{300})_{5,y}$.

In [ ]:
unabsorbed = np.isin(paths[:, -1], TRANS).mean()
Q045 = P045[np.ix_(TRANS, TRANS)]
p_tail_exact = matrix_power(Q045, N_STEPS).sum(axis=1)[4]        # P_5(T > 300), 정확값
print(f"unabsorbed after {N_STEPS} steps: {unabsorbed}   (exact P_5(T>{N_STEPS}) = {p_tail_exact:.2e})")
assert unabsorbed < 1e-3

est_win045 = mc_proportion(paths[:, -1] == N_GR)
est_dur045 = mc_estimate(T045)
print(f"P_5(win) hat = {est_win045}")
print(f"E_5[T]   hat = {est_dur045}   (sd of T = {T045.std(ddof=1):.2f})")

$p=\tfrac12$ 사슬도 같은 방식으로 ($P_5(T>300)\approx3.7\times10^{-7}$).

In [ ]:
paths_half = simulate_chain(P05, 5, N_STEPS, rng, n_paths=N_PATHS)
T05 = hitting_times(paths_half, {0, N_GR})
p_tail_half = matrix_power(P05[np.ix_(TRANS, TRANS)], N_STEPS).sum(axis=1)[4]
print(f"unabsorbed (p=0.5): {np.isin(paths_half[:, -1], TRANS).mean()}   (exact {p_tail_half:.2e})")
assert np.isfinite(T05).all()
est_win05 = mc_proportion(paths_half[:, -1] == N_GR)
est_dur05 = mc_estimate(T05)
print(f"P_5(win) hat = {est_win05}   E_5[T] hat = {est_dur05}   (sd = {T05.std(ddof=1):.2f})")

$T$의 정확한 분포는 행렬 거듭제곱으로 얻는다: $s_k=P_5(T>k)=\sum_{y\in T}(P^k)_{5,y}$이므로 $P_5(T=k)=s_{k-1}-s_k$. $T$의 분산은 $E[T^2]=((2N-I)t)_i$에서 나온다 (연습: $T=\sum_jV_j$와 $E_i[V_j^2]$로 유도해 보라).

In [ ]:
def absorption_time_pmf(P, x0, transient, k_max):
    """helper (spkit에 없어서 여기서 정의): P_{x0}(T = k), k = 1..k_max,
    from s_k = sum_{y in transient} (P^k)[x0, y] = P_{x0}(T > k)."""
    P = np.asarray(P, dtype=float)
    Pk = np.eye(len(P))
    s = np.empty(k_max + 1)
    for k in range(k_max + 1):
        s[k] = Pk[x0, transient].sum()
        Pk = Pk @ P
    return np.arange(1, k_max + 1), s[:-1] - s[1:]

K_MAX = 120
ks, pmf045 = absorption_time_pmf(P045, 5, TRANS, K_MAX)
N045, t045 = res045["N"], res045["t"]
var045 = ((2 * N045 - np.eye(len(TRANS))) @ t045 - t045**2)[4]        # Var_5(T)
print(f"pmf mass on k<=120: {pmf045.sum():.4f}, mean from pmf: {(ks * pmf045).sum():.3f}, "
      f"exact mean {t045[4]:.3f}, exact sd {np.sqrt(var045):.2f}")
ax = plot_hist_vs_pmf(np.minimum(T045, K_MAX), ks, pmf045, label_pmf="exact pmf (matrix powers)")
ax.axvline(t045[4], color="C3", ls="--", lw=1, label=f"exact mean {t045[4]:.2f}")
ax.set_xlabel(f"absorption time T (values > {K_MAX} pooled)"); ax.set_ylabel("probability"); ax.legend()
ax.set_title(f"Duration of the game from i=5, p=0.45: mean {t045[4]:.2f}, sd {np.sqrt(var045):.1f}")
plt.show()

### 4.3 작은 흡수 사슬: 패턴 HH의 대기시간

공정한 동전을 던져 처음으로 앞면이 두 번 연속(HH) 나올 때까지의 던짐 횟수. 상태를 '패턴 진행 정도'로 잡으면 3-상태 흡수 사슬이 된다: 0 = 진행 없음(마지막이 T이거나 시작), 1 = 마지막이 H, 2 = HH 달성(흡수).
$$P_{HH}=\begin{pmatrix}1/2&1/2&0\\1/2&0&1/2\\0&0&1\end{pmatrix}.$$
1단계 분석: $t_0=1+\tfrac12t_0+\tfrac12t_1$, $t_1=1+\tfrac12t_0$ (상태 1에서 T가 나오면 상태 0으로 **완전히** 되돌아간다). 05c에서는 같은 답을 마팅게일(ABRACADABRA 논법)로 다시 얻는다.

In [ ]:
P_HH = np.array([[0.5, 0.5, 0.0],
                 [0.5, 0.0, 0.5],
                 [0.0, 0.0, 1.0]])
res_HH = absorption(P_HH, [0, 1], [2])
t_HH_exact = float(res_HH["t"][0])
var_HH = ((2 * res_HH["N"] - np.eye(2)) @ res_HH["t"] - res_HH["t"]**2)[0]
print("N_HH =", res_HH["N"].tolist(), "  t =", res_HH["t"], f"  Var_0(T) = {var_HH:.1f}")

N_STEPS_HH = 200
p_tail_HH = matrix_power(P_HH[:2, :2], N_STEPS_HH).sum(axis=1)[0]
paths_HH = simulate_chain(P_HH, 0, N_STEPS_HH, rng, n_paths=N_PATHS)
T_HH = hitting_times(paths_HH, 2)
print(f"unabsorbed after {N_STEPS_HH} tosses: {np.isinf(T_HH).mean()}   (exact P(T>{N_STEPS_HH}) = {p_tail_HH:.1e})")
assert np.isfinite(T_HH).all()
est_t_HH = mc_estimate(T_HH)
print(f"E[T_HH] hat = {est_t_HH}")

In [ ]:
ks_HH, pmf_HH = absorption_time_pmf(P_HH, 0, [0, 1], 30)
ax = plot_hist_vs_pmf(np.minimum(T_HH, 30), ks_HH[1:], pmf_HH[1:], label_pmf="exact pmf")   # k = 2..30
ax.axvline(t_HH_exact, color="C3", ls="--", lw=1, label=f"exact mean {t_HH_exact:.0f}")
ax.set_xlabel("number of tosses until HH (values > 30 pooled)"); ax.set_ylabel("probability"); ax.legend()
ax.set_title(f"Waiting time for HH: E = {t_HH_exact:.0f}, sd = {np.sqrt(var_HH):.2f}")
plt.show()

### 4.4 01b의 7-상태 사슬: 닫힌 클래스가 여러 상태일 때

01b의 사슬을 다시 정의한다. 일시 집합 $\{2,3\}$, 닫힌 클래스 $\{0,1\}$과 $\{4,5,6\}$. `absorption`의 `absorbing`에는 닫힌 클래스의 상태를 **모두** 넣는다(함정 5). $B$의 열은 개별 상태 $0,1,4,5,6$이고 $B_{ia}=P_i(X_T=a)$는 '닫힌 클래스에 **들어가는 순간**의 상태가 $a$'일 확률이므로, 2·3에서 진입할 수 없는 열(1, 5, 6)은 0이다 — 그 뒤 사슬이 1이나 5를 무한히 방문해도 상관없다. 클래스별 흡수확률은 열을 합쳐 읽는다.

In [ ]:
P7 = np.array([
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0],   # 0: 닫힌 클래스 {0,1}
    [0.3, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0],   # 1
    [0.2, 0.0, 0.3, 0.5, 0.0, 0.0, 0.0],   # 2: 일시 클래스 {2,3}, 0 으로 샘
    [0.0, 0.0, 0.4, 0.2, 0.4, 0.0, 0.0],   # 3: 4 로 샘
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],   # 4: 순환 4 -> 5 -> 6 -> 4
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],   # 5
    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],   # 6
])
T7, A7 = [2, 3], [0, 1, 4, 5, 6]
assert is_stochastic(P7) and classify_states(P7)["transient_states"] == T7
res7 = absorption(P7, T7, A7)
print("N  (rows/cols = 2,3):        ", np.round(res7["N"], 4).tolist())
print("B  (rows = 2,3; cols = 0,1,4,5,6):")
for i, row in zip(T7, res7["B"]):
    print(f"   from {i}: {np.round(row, 4).tolist()}   -> P(class {{0,1}}) = {row[:2].sum():.4f}, "
          f"P(class {{4,5,6}}) = {row[2:].sum():.4f}")
print("t  (from 2, 3):              ", np.round(res7["t"], 4).tolist())
p_class01_exact = float(res7["B"][0, :2].sum())     # 4/9
t_exit_exact = float(res7["t"][0])                  # 65/18
assert np.isclose(res7["B"].sum(1), 1).all()        # 행합 1: 닫힌 클래스를 빠짐없이 넣었다

MC: 2에서 출발, 지평 100. 앞에서처럼 정확값 $P_2(T>100)=\sum_{y\in T}(Q^{100})_{2,y}$를 행렬 거듭제곱으로 계산해 지평이 충분한지 확인한 뒤, 클래스 $\{0,1\}$에 흡수된 비율과 $\{2,3\}$을 떠난 시각의 평균을 잰다.

In [ ]:
N_STEPS_7 = 100
p_tail_7 = matrix_power(P7[np.ix_(T7, T7)], N_STEPS_7).sum(axis=1)[0]     # P_2(T > 100), 정확값
paths7 = simulate_chain(P7, 2, N_STEPS_7, rng, n_paths=N_PATHS)
T7_exit = hitting_times(paths7, set(A7))
print(f"unabsorbed after {N_STEPS_7} steps: {np.isinf(T7_exit).mean()}   (exact P_2(T>{N_STEPS_7}) = {p_tail_7:.1e})")
assert np.isfinite(T7_exit).all()
est_class01 = mc_proportion(np.isin(paths7[:, -1], [0, 1]))
est_t_exit = mc_estimate(T7_exit)
print(f"P_2(absorbed in {{0,1}}) hat = {est_class01}")
print(f"E_2[exit time]          hat = {est_t_exit}")

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 코드로 계산했다: 도박꾼 파산은 `gamblers_ruin_prob`/`gamblers_ruin_duration`(정리 3), 나머지는 `absorption`의 $B=NR$, $t=N\mathbf 1$(정리 2). $N_{55}$ 행은 MC가 아니므로 SE $=0$으로 둔다.

In [ ]:
p_win_exact = gamblers_ruin_prob(5, N_GR, 0.45)
dur045_exact = gamblers_ruin_duration(5, N_GR, 0.45)
dur05_exact = gamblers_ruin_duration(5, N_GR, 0.5)
n55_exact = float(N_half_formula[4, 4])                     # 2*5*5/10

rows = [
    {"name": "P_5(win), p=0.45", "predicted": predictions["p_win_045"], "estimate": est_win045, "exact": p_win_exact},
    {"name": "E_5[T], p=0.45", "predicted": predictions["duration_045"], "estimate": est_dur045, "exact": dur045_exact},
    {"name": "E_5[T], p=0.5", "predicted": None, "estimate": est_dur05, "exact": dur05_exact},
    {"name": "N_55, p=0.5 (not MC)", "predicted": predictions["n55_half"], "estimate": float(res05["N"][4, 4]), "exact": n55_exact, "se": 0.0},
    {"name": "E[T_HH]", "predicted": predictions["t_HH"], "estimate": est_t_HH, "exact": t_HH_exact},
    {"name": "P_2(class {0,1}), 7-state", "predicted": predictions["p_class01_from_2"], "estimate": est_class01, "exact": p_class01_exact},
    {"name": "E_2[exit time], 7-state", "predicted": predictions["t_exit_from_2"], "estimate": est_t_exit, "exact": t_exit_exact},
]
print(f"exact: h(5)={p_win_exact:.6f}, tau(5)={dur045_exact:.6f}, tau_half(5)={dur05_exact:.1f}, "
      f"t_HH={t_HH_exact:.1f}, P_2(class01)={p_class01_exact:.6f}, t_exit={t_exit_exact:.6f}")
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P_5(win)**: 어긋나면 $B=NR$의 열 순서(0 | 10)를 잘못 읽었거나 지평 끝에 미흡수 경로가 남은 것. 00a의 시뮬레이션 값과도 같아야 한다.
- **E_5[T] (두 행)**: MC가 작게 나오면 지평 절단(함정 2); 크게 어긋나면 `hitting_times`에 흡수 집합 $\{0,10\}$ 대신 하나만 넣은 것.
- **N_55**: 정확식끼리의 비교. 다르면 대칭 걷기의 그린 함수 공식 $2\min(i,j)(N-\max(i,j))/N$을 잘못 쓴 것(SE 0이므로 아주 작은 차이도 원인을 찾아야 한다).
- **E[T_HH]**: 어긋나면 $P_{HH}$의 상태 1에서 T가 나올 때 상태 0으로 돌아가는 전이를 빠뜨린 것 — E2의 HT와 혼동하기 쉽다.
- **P_2(class {0,1})**: 어긋나면 `absorbing`에 $\{4,5,6\}$의 일부만 넣어 $B$의 행합이 1이 안 된 것(함정 5).
- **E_2[exit time]**: 어긋나면 `hitting_times`의 목표를 '닫힌 상태 전체'가 아니라 일부로 잡은 것.

In [ ]:
assert_close(est_win045, p_win_exact, k=4, name="p_win_045")
assert_close(est_dur045, dur045_exact, k=4, name="duration_045")
assert_close(est_dur05, dur05_exact, k=4, name="duration_half")
assert_close(est_t_HH, t_HH_exact, k=4, name="t_HH")
assert_close(est_class01, p_class01_exact, k=4, name="p_class01_from_2")
assert_close(est_t_exit, t_exit_exact, k=4, name="t_exit_from_2")
print("all checks passed")

## 6. 연습문제

### E1 계산

01b 사슬의 일시 블록 $Q=\begin{pmatrix}0.3&0.5\\0.4&0.2\end{pmatrix}$ (행·열 = 상태 2, 3)에서 $R$의 열을 클래스별로 합쳐 $R'=\begin{pmatrix}0.2&0\\0&0.4\end{pmatrix}$ (열: $\{0,1\}$, $\{4,5,6\}$)라 하자. **손으로** $N=(I-Q)^{-1}$, $t=N\mathbf 1$, $B=NR'$를 구하시오. 그런 다음 4.4의 `res7`과 비교하시오 — 클래스를 뭉쳐도(lumping) 흡수확률·시간이 같은지.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$I-Q=\begin{pmatrix}0.7&-0.5\\-0.4&0.8\end{pmatrix}$, 행렬식 $0.56-0.20=0.36$. $2\times2$ 역행렬 공식으로
$$N=\frac1{0.36}\begin{pmatrix}0.8&0.5\\0.4&0.7\end{pmatrix}=\begin{pmatrix}20/9&25/18\\10/9&35/18\end{pmatrix}\approx\begin{pmatrix}2.2222&1.3889\\1.1111&1.9444\end{pmatrix}.$$
$t=N\mathbf 1=(20/9+25/18,\ 10/9+35/18)=(65/18,\ 55/18)\approx(3.6111,\ 3.0556)$.
$B=NR'$: 첫 행 $(20/9\cdot0.2,\ 25/18\cdot0.4)=(4/9,\ 5/9)$, 둘째 행 $(10/9\cdot0.2,\ 35/18\cdot0.4)=(2/9,\ 7/9)$. 행합이 1인지 확인: ✓.

읽기: $N_{22}=20/9$는 01b의 $E_2V_2=1/(1-\rho_{22})=1/0.45$와 같다. 2에서 출발하면 3을 평균 $25/18\approx1.39$번 지나간다.

검산 코드:
```python
Q = np.array([[0.3, 0.5], [0.4, 0.2]])
Rp = np.array([[0.2, 0.0], [0.0, 0.4]])
N = np.linalg.inv(np.eye(2) - Q)
print(N * 18)                      # [[40, 25], [20, 35]]  -> N = [[20/9, 25/18], [10/9, 35/18]]
print(N.sum(1) * 18)               # [65, 55]
print(N @ Rp * 9)                  # [[4, 5], [2, 7]]
# 뭉치지 않은 4.4 의 결과와 비교: 열을 클래스별로 합치면 같다
B_lumped = np.column_stack([res7["B"][:, :2].sum(1), res7["B"][:, 2:].sum(1)])
assert np.allclose(N, res7["N"]) and np.allclose(N @ Rp, B_lumped) and np.allclose(N.sum(1), res7["t"])
```

</details>

### E2 유도 후 시뮬레이션 검증

패턴 **HT**(앞 뒤)의 기대 대기시간을 1단계 분석으로 유도하고 HH($=6$)와 비교하시오. 왜 다른가? 3-상태 흡수 사슬 $P_{HT}$를 만들어 `absorption`과 `simulate_chain`(`N_PATHS` 경로, 지평 200)으로 확인하고 `assert_close`로 4 SE 안에서 일치하는지 검사하시오. 보너스: $E[T^2]=((2N-I)t)_0$으로 분산을 구해 시뮬레이션의 표본분산과 비교하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

상태 0 (진행 없음), 1 (마지막이 H), 2 (HT 달성). 상태 1에서 H가 또 나오면 '마지막이 H'는 그대로이므로 상태 1에 **머문다**:
$$P_{HT}=\begin{pmatrix}1/2&1/2&0\\0&1/2&1/2\\0&0&1\end{pmatrix}.$$
1단계 분석: $t_1=1+\tfrac12t_1\Rightarrow t_1=2$; $t_0=1+\tfrac12t_0+\tfrac12t_1\Rightarrow t_0=2+t_1=4$. 즉 $E[T_{HT}]=4<6=E[T_{HH}]$.

왜 다른가: HH는 상태 1에서 실패(T)하면 상태 0으로 **완전히 되돌아가** 진행이 사라지지만, HT는 상태 1에서 실패(H)해도 '마지막이 H'라는 진행이 **유지**된다. 같은 이유로 HH는 자기 자신과 겹칠 수 있는(overlapping) 패턴이라 대기시간이 길다 — 05c의 마팅게일 논법에서는 이것이 '겹침 항' $2+4=6$ vs $4$로 나타난다.

분산: $N=(I-Q)^{-1}=\begin{pmatrix}2&2\\0&2\end{pmatrix}$, $t=(4,2)$, $E_0[T^2]=((2N-I)t)_0=(3\cdot4+4\cdot2)=20$, $\operatorname{Var}_0T=20-16=4$, sd $2$. 20000경로면 SE $\approx2/\sqrt{20000}\approx0.014$.

코드:
```python
P_HT = np.array([[0.5, 0.5, 0.0], [0.0, 0.5, 0.5], [0.0, 0.0, 1.0]])
res_HT = absorption(P_HT, [0, 1], [2])
print(res_HT["t"])                                            # [4. 2.]
var_HT = ((2 * res_HT["N"] - np.eye(2)) @ res_HT["t"] - res_HT["t"]**2)[0]
paths_HT = simulate_chain(P_HT, 0, 200, rng, n_paths=N_PATHS)
T_HT = hitting_times(paths_HT, 2)
assert np.isfinite(T_HT).all()
est_HT = mc_estimate(T_HT)
t_HT_exact = float(res_HT["t"][0])                             # 4.0
print(est_HT, "exact", t_HT_exact, ";  var", T_HT.var(ddof=1), "exact", var_HT)   # ~4.0 ± 0.014, var ~4
assert_close(est_HT, t_HT_exact, k=4, name="t_HT")
```

</details>

### E3 가정을 깨보기

(a) $p=\tfrac12$, $N=10$, $i=5$에서 지평을 `n_steps=30`으로만 잡고 시뮬레이션한 뒤, `hitting_times`가 `inf`인 경로를 30으로 바꿔 평균을 내면 $E[T]$ 추정값은 어떻게 되는가? 정확값 $P_5(T>30)$과 $E[\min(T,30)]=\sum_{k=0}^{29}P_5(T>k)$를 행렬 거듭제곱으로 계산해 비교하시오.
(b) `absorption(P05, transient=list(range(0, 10)), absorbing=[10])`처럼 흡수 상태 0을 일시 집합에 잘못 넣으면 무슨 일이 생기는가? 정리 1의 어느 가정이 깨졌는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $Q=P[T,T]$의 거듭제곱으로 $P_5(T>30)=\sum_y(Q^{30})_{5y}\approx0.2873$ — 경로의 29%가 30걸음 뒤에도 안 끝났다. 절단 평균의 정확값은 꼬리합 공식 $E[\min(T,30)]=\sum_{k=0}^{29}P_5(T>k)\approx19.27$로, 참값 $25$보다 한참 작다. 지평 절단은 언제나 **아래로** 편향된다($\min(T,n)\le T$). 그래서 흡수 안 된 비율을 항상 출력하고, $P(T>n)$이 무시할 만할 때까지($300$걸음이면 $3.7\times10^{-7}$) 지평을 늘려야 한다. ($p=0.45$이면 $P_5(T>30)\approx0.2534$, $E[\min(T,30)]\approx18.58$ vs 참값 $23.17$.)

```python
Q05 = P05[np.ix_(TRANS, TRANS)]
p_tail_30 = matrix_power(Q05, 30).sum(1)[4]                          # 0.2873
E_min_30 = sum(matrix_power(Q05, k).sum(1)[4] for k in range(30))    # 19.27
paths_short = simulate_chain(P05, 5, 30, rng, n_paths=N_PATHS)
T_short = hitting_times(paths_short, {0, 10})
print("unabsorbed fraction:", np.isinf(T_short).mean(), " exact", p_tail_30)
est_trunc = mc_estimate(np.where(np.isinf(T_short), 30, T_short))
print(est_trunc, " exact E[min(T,30)] =", E_min_30, " true E[T] = 25")
assert_close(est_trunc, E_min_30, k=4, name="truncated mean")        # 절단 평균 자체는 정확히 예측된다
```

(b) `transient=[0,...,9]`이면 $Q$에 흡수 상태 0의 행 $(1,0,\dots,0)$이 들어간다. $Q$가 고유값 1을 가지므로 $I-Q$는 특이행렬: 이 예에서는 정확히 특이라 `np.linalg.inv`가 `LinAlgError: Singular matrix`를 낸다. (부동소수점 오차로 아슬아슬하게 가역이 되는 경우에는 $10^{15}$ 규모의 쓰레기 값이 나오므로 더 위험하다.) 깨진 가정은 정리 1의 '$T$는 일시 상태만 포함' — $\{0\}$은 닫힌 클래스라 $Q^n\not\to0$이고 급수 $\sum Q^k$가 발산한다. 확률적으로도 당연하다: 0에서 출발하면 '0의 방문 횟수'가 무한대다.

```python
try:
    absorption(P05, list(range(0, 10)), [10])
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)
Q_bad = P05[:10, :10]
print("eigenvalues of Q_bad closest to 1:", np.sort(np.abs(np.linalg.eigvals(Q_bad)))[-2:])   # ..., 1.0
print("classify_states says transient =", classify_states(P05)["transient_states"])           # [1, ..., 9]
```

</details>

## 7. 정리

1. 정준형 $P=\begin{pmatrix}Q&R\\0&I\end{pmatrix}$; 기본행렬 $N=(I-Q)^{-1}=\sum_kQ^k$, $N_{ij}$ = 흡수 전 $j$ 방문 횟수의 기댓값. 유한 사슬에서 $T$가 일시 상태만 포함하면 $Q^n\to0$이라 존재한다.
2. 흡수확률 $B=NR$, 기대 흡수시간 $t=N\mathbf 1$ — 둘 다 1단계 분석의 선형방정식 $(I-Q)B=R$, $(I-Q)t=\mathbf 1$의 유일해. 확률 방정식에는 상수항이 없고 시간 방정식에는 $1$이 붙는다.
3. 도박꾼 파산: $h(i)=(1-r^i)/(1-r^N)$, $\tau(i)=(i-N\,h(i))/(q-p)$, $r=q/p$; $p=\tfrac12$이면 $i/N$, $i(N-i)$. $N=10,\ p=0.45,\ i=5$: 승률 $0.2683$, 게임 길이 $23.17$걸음 — 00a의 시뮬레이션, 닫힌 공식, `absorption` 세 가지가 일치했다.
4. 작은 흡수 사슬로 패턴 대기시간도 푼다: $E[T_{HH}]=6$, $E[T_{HT}]=4$ — 실패했을 때 진행이 남는가가 차이다.
5. 시뮬레이션 지평 절단은 $E[T]$를 과소추정하고($E[\min(T,30)]=19.27$ vs $25$), 일시 집합에 재귀 상태를 넣으면 $I-Q$가 특이행렬이 된다.

**trap 카드**
- Q: `hitting_times`가 `inf`를 돌려준 경로를 '지평 값'으로 바꿔 평균 내면 $E[T]$ 추정은 어느 쪽으로 치우치나? 어떻게 알아채나?
- A: 아래로(과소추정) — $E[\min(T,n)]<E[T]$. 흡수 안 된 경로의 비율을 항상 출력하고, 행렬 거듭제곱으로 $P(T>n)=\sum_{y\in T}(Q^n)_{x_0y}$가 무시할 만한지($<10^{-6}$) 확인한 뒤 지평을 정한다.

**다음 노트북: 01d** — 흡수 사슬은 결국 멈춘다. 기약 사슬은 영원히 돌아다니는데, 그럼 '오랜 시간 뒤 어디에 있는가'의 답이 정상분포 $\pi$다. 01a에서 본 날씨 사슬의 $(4/7,\,3/7)$이 무엇인지 세 가지 방법(선형방정식·고유벡터·행렬 거듭제곱)으로 계산한다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 01c 흡수확률과 도박꾼의 파산 | 소요 __분 | 예측 적중 _/6 | 막힌 곳: ___ | 다음: 01d`